In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Load the combined weekly load + temperature features
feature_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "weekly_load_temperature_features.csv",
    parse_dates=["date"],
    index_col="date",
)

print(feature_df.head())
print(feature_df.columns)

              load_gw  temp_mean  temp_min  temp_max  heating_degree  \
date                                                                   
2015-01-04  47.233740   3.000000       3.0       3.0            12.5   
2015-01-11  56.191101   3.885714       1.2       8.5            81.3   
2015-01-18  57.672679   4.900000      -0.8       9.2            74.2   
2015-01-25  58.613304   0.028571      -0.7       0.9           108.3   
2015-02-01  58.734030   1.414286      -0.1       2.8            98.6   

            cooling_degree  
date                        
2015-01-04             0.0  
2015-01-11             0.0  
2015-01-18             0.0  
2015-01-25             0.0  
2015-02-01             0.0  
Index(['load_gw', 'temp_mean', 'temp_min', 'temp_max', 'heating_degree',
       'cooling_degree'],
      dtype='str')


In [2]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [3]:
# Example for SARIMAX, the temperature (and holiday) variables become exogenous regressors


y = feature_df["load_gw"]

X = feature_df[
    [
       # "holiday_days",
       # "has_holiday",
        "temp_mean",
        "heating_degree",
        "cooling_degree",
    ]
]

test_weeks = 104

y_train = y.iloc[:-test_weeks]
y_test = y.iloc[-test_weeks:]

X_train = X.iloc[:-test_weeks]
X_test = X.iloc[-test_weeks:]

sarimax_x = SARIMAX(
    y_train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 52),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarimax_x_fit = sarimax_x.fit(disp=False)

sarimax_x_fc = sarimax_x_fit.get_forecast(
    steps=len(y_test),
    exog=X_test,
)

sarimax_x_mean = sarimax_x_fc.predicted_mean
sarimax_x_mean.index = y_test.index

/Users/indhureddy/Documents/electricity-demand-forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/indhureddy/Documents/electricity-demand-forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/indhureddy/Documents/electricity-demand-forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/Users/indhureddy/Documents/electricity-demand-forecasting/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: Convergenc